# compresskit — GPU validation notebook (Kaggle or Colab)

Checks that the large-model changes really work on a **CUDA GPU**, then runs a short end-to-end compression with timings.
Those code paths (fp16 AMP, auto batch-size probe, Taylor pruning on GPU, per-step checkpoints) can only be tested on a GPU.

| # | Check | What it proves |
|---|-------|----------------|
| 1 | unit tests | the test-suite passes on this machine |
| 2 | AMP speed | fp16 autocast runs and is faster than fp32 (`img/s` for both) |
| 3 | batch probe | `fit_batch_size` finds a batch that fits, and leaves BatchNorm statistics untouched |
| 4 | data path | 32→224 GPU upscaling feeds the model (`img/s`) |
| 5 | Taylor pruning | importance backward pass runs on the GPU without OOM |
| 6 | end to end | short head fine-tune → gradual prune + fine-tune → INT8 → ONNX, a checkpoint saved per step |

**Kaggle:** *Accelerator → GPU T4*, *Internet → On*, optionally *Add Input → CIFAR-10* (python-pickle, has `cifar-10-batches-py`).
**Colab:** *Runtime → Change runtime type → T4 GPU*. CIFAR-10 downloads (170 MB).

This is a **mechanics test, not a benchmark**: the end-to-end part trains the new head for only a few hundred batches
and fine-tunes for a few batches per step, so its accuracy numbers show the pipeline works, not the final quality.
Change `ARCH` to test another model (`tv:resnet152`, `tv:densenet121`, `tv:convnext_tiny`, `tv:vgg16_bn`).

In [ ]:
# ======================= CONFIG =======================
ARCH           = "tv:resnet50"   # try: tv:resnet152, tv:densenet121, tv:convnext_tiny, tv:vgg16_bn
REQUIRE_GPU    = True            # stop at once if no CUDA GPU is attached
PRUNE_RATIO    = 0.5
STEPS          = 2               # gradual pruning steps
HEAD_BATCHES   = 150             # batches used to train the new 10-class head
FT_BATCHES     = 40              # training batches per fine-tune epoch after each prune step
REQUESTED_BATCH = 64             # effective training batch; the probe may split it into micro-batches
TEST_IMAGES    = 500             # test images used to score every stage
CALIB          = 128             # calibration images (BN, INT8, Taylor)
# ======================================================

In [ ]:
import os, subprocess, sys, json, shutil, time, copy, itertools, pathlib
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
WORK = "/content" if IN_COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
print("platform:", "Colab" if WORK == "/content" else "Kaggle", "| working dir:", WORK)
REPO = "https://github.com/Bhagyesh0603/Model-Compression-ToolKit..git"
if not os.path.exists(f"{WORK}/ck"):
    subprocess.run(["git", "clone", "-q", REPO, f"{WORK}/ck"], check=True)
os.chdir(f"{WORK}/ck")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
CUDA = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if CUDA else "NONE")
print("torch", torch.__version__, "| cuda", torch.version.cuda)
if REQUIRE_GPU:
    assert CUDA, "No CUDA GPU. Kaggle: Accelerator -> GPU T4.  Colab: Runtime -> Change runtime type -> T4 GPU. Then Restart & Run All."
assert os.path.exists("compresskit/core/amp.py"), "this clone is missing the large-model changes - push them to GitHub first"

In [ ]:
from compresskit.core import amp
from compresskit.core.host import compute_device
from compresskit.core.memory import fit_batch_size, split_batch
from compresskit.zoo.datasets import get_bundle
from compresskit.zoo.torchvision_models import build_torchvision, replace_head

DEV = compute_device()
RESULTS = {}

def check(name, ok, detail=""):
    RESULTS[name] = {"pass": bool(ok), "detail": detail}
    print(("PASS " if ok else "FAIL ") + name + (f"  --  {detail}" if detail else ""), flush=True)

def sync():
    if CUDA: torch.cuda.synchronize()

def peak_gb():
    return torch.cuda.max_memory_allocated() / 1e9 if CUDA else 0.0

def reset_peak():
    if CUDA: torch.cuda.reset_peak_memory_stats()

def fresh():
    return replace_head(build_torchvision(ARCH), 10)

class Few:
    """First n batches of a loader, with the attributes the pipeline reads."""
    def __init__(self, loader, n): self.loader, self.n = loader, n
    def __len__(self): return min(self.n, len(self.loader))
    batch_size = property(lambda s: s.loader.batch_size)
    dataset = property(lambda s: s.loader.dataset)
    def __iter__(self): return itertools.islice(iter(self.loader), self.n)

bundle = get_bundle("cifar10-upscaled", None, 224, TEST_IMAGES, 0)
print("device for compute:", DEV)

## 1 · Unit tests on this machine

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-x", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(r.stdout[-600:])
check("1 unit tests", r.returncode == 0, r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-200:])

## 2 · fp16 AMP vs fp32 training speed (needs CUDA)

In [ ]:
def train_rate(model, use_amp, batch=16, iters=8):
    amp.prepare_model(model, DEV).train()
    opt = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
    scaler = amp.Scaler(DEV, use_amp)
    x = amp.prepare_batch(torch.randn(batch, 3, 224, 224), DEV)
    y = torch.randint(0, 10, (batch,), device=DEV)
    def one():
        with amp.autocast(DEV, use_amp):
            out = model(x)
        loss = torch.nn.functional.cross_entropy(out.float(), y)
        opt.zero_grad(set_to_none=True); scaler.backward(loss); scaler.step(opt)
        return loss
    for _ in range(3): one()
    sync(); reset_peak(); t = time.perf_counter()
    for _ in range(iters): loss = one()
    sync()
    assert torch.isfinite(loss), "loss is not finite"
    return batch * iters / (time.perf_counter() - t), peak_gb()

if CUDA:
    try:
        fp32_rate, fp32_mem = train_rate(fresh(), False)
        amp_rate, amp_mem = train_rate(fresh(), True)
        print(f"fp32: {fp32_rate:7.1f} img/s  peak {fp32_mem:.2f} GB")
        print(f"amp : {amp_rate:7.1f} img/s  peak {amp_mem:.2f} GB   -> {amp_rate / fp32_rate:.2f}x speed, {fp32_mem / max(amp_mem, 1e-9):.2f}x less memory")
        check("2 AMP training step", True, f"{amp_rate / fp32_rate:.2f}x faster, {fp32_mem / max(amp_mem, 1e-9):.2f}x less memory")
    except Exception as e:
        check("2 AMP training step", False, repr(e)[:200])
else:
    print("skipped: no CUDA"); check("2 AMP training step", True, "skipped (no CUDA)")

## 3 · Auto batch size probe

In [ ]:
model = fresh()
before = {k: v.clone() for k, v in model.state_dict().items()}
t = time.perf_counter()
fitted = fit_batch_size(model, bundle.input_shape, start=256, device=DEV, teacher=model)
micro, accum = split_batch(REQUESTED_BATCH, fitted)
model.cpu()
bn_same = all(torch.equal(v, before[k]) for k, v in model.state_dict().items())
print(f"largest batch that fits at 224x224: {fitted}   ({time.perf_counter() - t:.1f}s)")
print(f"requested effective batch {REQUESTED_BATCH} -> micro-batch {micro} x {accum} accumulation steps")
check("3 batch probe", fitted >= 1 and bn_same, f"fits {fitted}; BN stats {'untouched' if bn_same else 'MODIFIED'}")
del model

## 4 · Data path (GPU upscaling)

In [ ]:
loader = bundle.train_loader(batch_size=micro)
n_batches = 30
it = iter(loader); next(it)
sync(); t = time.perf_counter(); seen = 0
for _ in range(n_batches):
    x, y = next(it); seen += len(x)
sync()
rate = seen / (time.perf_counter() - t)
print(f"train loader: {rate:.0f} img/s   batch {tuple(x.shape)} on {x.device}")
check("4 data path", x.shape[-1] == 224 and rate > 0, f"{rate:.0f} img/s, batch on {x.device}")
del it, loader

## 5 · Taylor pruning on the GPU

In [ ]:
from compresskit.pruning.prune import prune
calib = bundle.calibration_loader(n=CALIB, batch_size=min(micro, 32))
net = fresh().cpu().eval()
reset_peak(); t = time.perf_counter()
try:
    pruned, stats = prune(net, torch.randn(*bundle.input_shape), ratio=PRUNE_RATIO, align=16, steps=1,
                          criterion="taylor", loader=calib, batches=2)
    removed = 1 - sum(p.numel() for p in pruned.parameters()) / sum(p.numel() for p in net.parameters())
    print(f"{stats.summary()}\n{time.perf_counter() - t:.1f}s, peak GPU memory {peak_gb():.2f} GB")
    check("5 Taylor pruning", removed > 0.3, f"{removed * 100:.0f}% params removed, peak {peak_gb():.2f} GB")
except Exception as e:
    check("5 Taylor pruning", False, repr(e)[:200])
del net

## 6 · End to end: head fine-tune → gradual prune + fine-tune → INT8 → ONNX
Mechanics only: the head is trained for `HEAD_BATCHES` batches and each fine-tune epoch uses `FT_BATCHES` batches.

In [ ]:
from compresskit.pipeline import Pipeline
from compresskit.zoo.train import TrainConfig, train

train_loader = bundle.train_loader(batch_size=micro)
test_loader = bundle.test_loader(batch_size=micro)
original = fresh()
t0 = time.perf_counter()
train(original, Few(train_loader, HEAD_BATCHES),
      TrainConfig(epochs=1, lr=0.01, warmup_epochs=0, device=DEV, amp=True, accum_steps=accum))
print(f"head fine-tune: {time.perf_counter() - t0:.0f}s")

work = f"{WORK}/validate_work"
shutil.rmtree(work, ignore_errors=True)
pipe = Pipeline(original, calib, test_loader, train_loader=Few(train_loader, FT_BATCHES),
                input_shape=bundle.input_shape, workdir=work, warmup=5, runs=20, device=DEV, accum_steps=accum)
pipe.prune(ratio=PRUNE_RATIO, align=16, steps=STEPS, criterion="taylor", fine_tune_epochs=1, loss="dkd")
pipe.quantize(precision="int8", allow_slow_int8=True)
print(pipe.preflight()); print()
t0 = time.perf_counter()
try:
    result = pipe.run()
    elapsed = time.perf_counter() - t0
    print(); print(result.summary()); print(f"pipeline wall time: {elapsed:.0f}s")
    ckpts = sorted(p.name for p in pathlib.Path(work).glob("prune_step*.pt"))
    b, f = result.baseline, result.final
    removed = 1 - f.parameters / b.parameters
    check("6 end to end", len(ckpts) == STEPS and removed > 0.5 and f.precision == "int8" and os.path.exists(f.artifact),
          f"acc {b.accuracy*100:.1f}% -> {f.accuracy*100:.1f}%, params -{removed*100:.0f}%, size {b.size_mb:.0f} -> {f.size_mb:.1f} MB, "
          f"{len(ckpts)} step checkpoints, {elapsed:.0f}s")
except Exception as e:
    import traceback; traceback.print_exc()
    check("6 end to end", False, repr(e)[:200])

## Summary

In [ ]:
print(f"{'check':<24}{'result':<8}detail"); print("-" * 90)
for name, r in RESULTS.items():
    print(f"{name:<24}{'PASS' if r['pass'] else 'FAIL':<8}{r['detail']}")
all_ok = all(r["pass"] for r in RESULTS.values())
print("\nALL CHECKS PASSED" if all_ok else "\nSOME CHECKS FAILED - copy this whole output and send it back")

info = {"arch": ARCH, "gpu": torch.cuda.get_device_name(0) if CUDA else None, "torch": torch.__version__,
        "platform": "colab" if WORK == "/content" else "kaggle", "results": RESULTS}
out = f"{WORK}/gpu_validation_results.json"
json.dump(info, open(out, "w"), indent=2)
print("saved", out)
if WORK == "/content":
    from google.colab import files
    files.download(out)